# Books Scraper

This notebook demonstrates scraping book data from [Books to Scrape](http://books.toscrape.com/) - another practice website for web scraping.

**Features covered:**
- Scraping product listings
- Extracting prices and ratings
- Handling relative URLs
- Data cleaning

In [1]:
from bs4 import BeautifulSoup
import requests
import pandas as pd
import re

In [2]:
base_url = "http://books.toscrape.com/"

response = requests.get(base_url)
print(f"Status Code: {response.status_code}")

soup = BeautifulSoup(response.text, 'html.parser')

Status Code: 200


In [3]:
# Map star ratings from class names to numbers
rating_map = {
    'One': 1,
    'Two': 2,
    'Three': 3,
    'Four': 4,
    'Five': 5
}

def get_rating(article):
    """Extract star rating from book article element."""
    star_element = article.find('p', class_='star-rating')
    for rating, value in rating_map.items():
        if rating in star_element.get('class', []):
            return value
    return 0

In [4]:
# Find all book containers
books = soup.find_all('article', class_='product_pod')
print(f"Found {len(books)} books on this page")

Found 20 books on this page


In [5]:
# Extract book data
books_data = []

for book in books:
    title = book.find('h3').find('a')['title']
    price = book.find('p', class_='price_color').text
    # Clean price - remove currency symbol and convert to float
    price_clean = float(price.replace('£', '').replace('Â', ''))
    availability = book.find('p', class_='instock availability').text.strip()
    rating = get_rating(book)
    link = base_url + book.find('h3').find('a')['href']

    books_data.append({
        'title': title,
        'price': price_clean,
        'availability': availability,
        'rating': rating,
        'url': link
    })

df = pd.DataFrame(books_data)
df.head(10)

,title,price,availability,rating,url
0,A Light in the Attic,51.77,In stock,3,http://books.toscrape.com/catalogue/a-light-in...
1,Tipping the Velvet,53.74,In stock,1,http://books.toscrape.com/catalogue/tipping-th...
2,Soumission,50.10,In stock,1,http://books.toscrape.com/catalogue/soumission...
3,Sharp Objects,47.82,In stock,4,http://books.toscrape.com/catalogue/sharp-obje...
4,Sapiens: A Brief History of Humankind,54.23,In stock,5,http://books.toscrape.com/catalogue/sapiens-a-...
5,The Requiem Red,22.65,In stock,1,http://books.toscrape.com/catalogue/the-requie...
6,The Dirty Little Secrets of Getting Your Dream...,33.34,In stock,4,http://books.toscrape.com/catalogue/the-dirty-...
7,The Coming Woman: A Novel Based on the Life of...,17.93,In stock,3,http://books.toscrape.com/catalogue/the-coming...
8,The Boys in the Boat: Nine Americans and Their...,22.60,In stock,4,http://books.toscrape.com/catalogue/the-boys-i...
9,The Black Maria,52.15,In stock,1,http://books.toscrape.com/catalogue/the-black-...


## Scrape All Pages

In [6]:
def scrape_all_books(max_pages=50):
    """Scrape books from multiple pages."""
    all_books = []

    for page in range(1, max_pages + 1):
        if page == 1:
            url = base_url
        else:
            url = f"{base_url}catalogue/page-{page}.html"

        response = requests.get(url)

        if response.status_code != 200:
            print(f"Reached end at page {page}")
            break

        soup = BeautifulSoup(response.text, 'html.parser')
        books = soup.find_all('article', class_='product_pod')

        if not books:
            break

        for book in books:
            title = book.find('h3').find('a')['title']
            price = book.find('p', class_='price_color').text
            price_clean = float(price.replace('£', '').replace('Â', ''))
            availability = book.find('p', class_='instock availability').text.strip()
            rating = get_rating(book)

            all_books.append({
                'title': title,
                'price': price_clean,
                'availability': availability,
                'rating': rating,
                'page': page
            })

        print(f"Scraped page {page} - Total books: {len(all_books)}")

    return all_books

# Scrape first 5 pages as demo
all_books = scrape_all_books(max_pages=5)

Scraped page 1 - Total books: 20
Scraped page 2 - Total books: 40
Scraped page 3 - Total books: 60
Scraped page 4 - Total books: 80
Scraped page 5 - Total books: 100


In [7]:
# Create DataFrame and analyze
df_all = pd.DataFrame(all_books)

print(f"Total books scraped: {len(df_all)}")
print(f"\nPrice Statistics:")
print(df_all['price'].describe())
print(f"\nRating Distribution:")
print(df_all['rating'].value_counts().sort_index())

Total books scraped: 100

Price Statistics:
count    100.000000
mean      34.560700
std       14.638531
min       10.160000
25%       19.897500
50%       34.775000
75%       47.967500
max       58.110000
Name: price, dtype: float64

Rating Distribution:
rating
1    22
2    19
3    22
4    18
5    19
Name: count, dtype: int64


In [8]:
# Export to CSV
df_all.to_csv('books_output.csv', index=False)
print("Data exported to books_output.csv")

Data exported to books_output.csv
